# The Most Efficient NBA Duo: Application
**MIS 433 Course Project | Sam Nocera & Jackson Hardy | Fall 2026**

This notebook turns our analysis into an application with three features. Each feature:
1. takes **input from the user**,
2. pulls data from **our scored dataset** (built in `01_analysis.ipynb`) and the **BALLDONTLIE API** (live NBA teams, players and games),
3. uses the **OpenAI API** to generate text, audio, or an image.

| Feature | User input | Dataset | BALLDONTLIE API | OpenAI output |
|---|---|---|---|---|
| 1. Team Duo Scouting Report | a team | team's top duos | 2026-27 record and recent games | written report + audio narration |
| 2. Duo vs Duo | two duos | both duos' stats | player profiles | written verdict |
| 3. Build Your Own Model | stat weights | all duos re-scored | team name | written explanation + award poster image |


## Setup

**API keys:** on Colab, open Secrets (key icon) in the left sidebar and add two secrets with notebook access turned on:
- `OpenAIAPI` (your OpenAI API key)
- `BalldontlieAPI` (free account at app.balldontlie.io)

In [ ]:
# Install the OpenAI package if you haven't yet
#!pip install --upgrade openai

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
import time
from sklearn.preprocessing import MinMaxScaler

In [ ]:
from google.colab import userdata
openai_api_key = userdata.get('OpenAIAPI')
bdl_api_key = userdata.get('BalldontlieAPI')

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=openai_api_key)

### Load our scored dataset

In [ ]:
data_url = 'https://raw.githubusercontent.com/samnocera05-ui/mis433/main/final-project/data/'

duos = pd.read_csv(data_url + 'duos_scored_2025_26.csv')        # 2025-26, scored in 01_analysis
duos_new = pd.read_csv(data_url + 'duos_clean_2026_27.csv')     # 2026-27 so far, cleaned
print(duos.shape, duos_new.shape)
duos.head()

### Helper functions

**OpenAI** (same pattern as our class notebook `openai-api-basics`): text, text-to-speech, and image generation.

In [ ]:
def get_completion(prompt, model="gpt-6-luna"):
    messages = [
        {"role": "system", "content": "You are an NBA analytics expert. Be specific, use the numbers you are given, and keep it short."},
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
    )
    return response.choices[0].message.content

In [ ]:
from IPython.display import Audio, Image, display
import base64

def text_to_speech(text, voice="onyx"):
    response = client.chat.completions.create(
        model="gpt-audio-1.5",
        modalities=["text", "audio"],
        audio={"voice": voice, "format": "wav"},
        messages=[
            {"role": "system", "content": "You are an NBA broadcast narrator. Read the user's text exactly as written. Do not add commentary."},
            {"role": "user", "content": text}
        ]
    )
    return base64.b64decode(response.choices[0].message.audio.data)

def generate_image(prompt):
    result = client.images.generate(
        model="gpt-image-2.5-flare",
        prompt=prompt,
        size="1024x1024",
        quality="low"
    )
    return base64.b64decode(result.data[0].b64_json)

**BALLDONTLIE API** (same pattern as our class notebook `api-basics`: `requests.get`, check `response.ok`, read `response.json()`).

The free tier allows 5 requests per minute, so the helper waits 12 seconds after each call.

In [ ]:
BDL_URL = 'https://api.balldontlie.io/v1/'

def bdl_get(endpoint, params=None):
    response = requests.get(BDL_URL + endpoint, headers={'Authorization': bdl_api_key}, params=params)
    time.sleep(12)   # free tier: 5 requests per minute
    if response.ok:
        return response.json()
    else:
        print(f"Request to {endpoint} failed with status code {response.status_code}")
        return None

In [ ]:
# Get all 30 teams once and keep them in a dataframe
data = bdl_get('teams')
teams = pd.DataFrame(data['data'])
teams = teams[teams['id'] <= 30]    # ids above 30 are historical franchises
teams[['id', 'abbreviation', 'full_name', 'conference', 'division']].head()

In [ ]:
def get_team_games(team_abbr, season=2026):
    # season=2026 means the 2026-27 season
    team_id = int(teams[teams['abbreviation'] == team_abbr]['id'].iloc[0])
    data = bdl_get('games', params={'team_ids[]': team_id, 'seasons[]': season, 'per_page': 100})
    games = []
    for g in data['data']:
        if g['status'] == 'Final':
            home = g['home_team']['abbreviation'] == team_abbr
            team_score = g['home_team_score'] if home else g['visitor_team_score']
            opp_score = g['visitor_team_score'] if home else g['home_team_score']
            opponent = g['visitor_team']['abbreviation'] if home else g['home_team']['abbreviation']
            games.append({'date': g['date'], 'opponent': opponent, 'home': home,
                          'team_score': team_score, 'opp_score': opp_score,
                          'result': 'W' if team_score > opp_score else 'L'})
    games = pd.DataFrame(games)
    if len(games) > 0:
        games = games.sort_values('date', ascending=False)
    return games

In [ ]:
# Player names in our data look like 'N. Jokić' or 'J. Smith Jr.'.
# BALLDONTLIE stores names without accents, so we replace accented letters first.
accents = {'ć': 'c', 'č': 'c', 'é': 'e', 'á': 'a', 'í': 'i', 'ó': 'o', 'ö': 'o', 'ü': 'u', 'š': 's', 'ž': 'z', 'ņ': 'n', 'ģ': 'g', 'ū': 'u'}

def get_player(short_name, team_abbr):
    for letter, plain in accents.items():
        short_name = short_name.replace(letter, plain)
    first_initial = short_name.split('. ')[0]
    last_name = short_name.split('. ')[1].split(' ')[0]     # 'Smith Jr.' -> 'Smith'

    data = bdl_get('players', params={'search': last_name, 'per_page': 100})
    if data is None:
        return None
    for p in data['data']:
        if p['first_name'].startswith(first_initial) and p['team'] is not None and p['team']['abbreviation'] == team_abbr:
            return {'name': p['first_name'] + ' ' + p['last_name'], 'position': p['position'], 'height': p['height'],
                    'draft_year': p['draft_year'], 'country': p['country']}
    # Player may have changed teams since 2025-26: fall back to the first name match
    for p in data['data']:
        if p['first_name'].startswith(first_initial):
            return {'name': p['first_name'] + ' ' + p['last_name'], 'position': p['position'], 'height': p['height'],
                    'draft_year': p['draft_year'], 'country': p['country'],
                    'current_team': p['team']['abbreviation'] if p['team'] is not None else 'free agent'}
    return None

## Feature 1: Team Duo Scouting Report

**Purpose:** a coach or fan picks a team and gets a scouting report on its most efficient duos, read aloud like a broadcast segment.

**How it works:**
1. **Input:** a team abbreviation
2. **Dataset:** the team's top 3 duos from 2025-26 (Balanced model) and its best duo so far in 2026-27
3. **BALLDONTLIE API:** the team's 2026-27 record and last 5 games
4. **OpenAI:** writes the report (text), then narrates it (audio)

In [ ]:
def team_scouting_report(team_abbr):
    # 1. Dataset: top 3 duos last season, best duo this season
    top3 = duos[duos['TEAM'] == team_abbr].sort_values('DES_Balanced', ascending=False).head(3)
    this_season = duos_new[duos_new['TEAM'] == team_abbr].sort_values('NetRtg', ascending=False).head(3)

    # 2. API: record and recent games
    games = get_team_games(team_abbr)
    wins = (games['result'] == 'W').sum()
    losses = (games['result'] == 'L').sum()
    team_name = teams[teams['abbreviation'] == team_abbr]['full_name'].iloc[0]

    print(f"{team_name}: {wins}-{losses} in 2026-27")
    print('\nTop 3 duos, 2025-26 (Balanced model):')
    display(top3[['Lineups', 'MIN', 'OffRtg', 'DefRtg', 'NetRtg', 'DES_Balanced']].round(1))
    print('Best duos so far in 2026-27 (100+ minutes):')
    display(this_season[['Lineups', 'MIN', 'NetRtg']])
    print('Last 5 games:')
    display(games.head(5))

    # 3. OpenAI: written report
    prompt = f"""
    Write a scouting report (about 150 words) on the {team_name}'s most efficient two-player duos.
    Duo Efficiency Score (DES) is 0-100; higher is better. Net Rating is points per 100 possessions.

    Top 3 duos last season (2025-26):
    {top3[['Lineups', 'MIN', 'OffRtg', 'DefRtg', 'NetRtg', 'DES_Balanced']].round(1).to_string(index=False)}

    Best duos so far this season (2026-27):
    {this_season[['Lineups', 'MIN', 'NetRtg']].to_string(index=False)}

    Record this season: {wins}-{losses}. Last 5 games:
    {games.head(5).to_string(index=False)}

    Say why the top duo works (offense, defense, or both), whether it is still producing this season,
    and one lineup recommendation for the coach.
    """
    report = get_completion(prompt)
    print('\n' + report)

    # 4. OpenAI: audio narration
    display(Audio(text_to_speech(report), autoplay=False))
    return report

In [ ]:
team = 'SAS'
# Or replace this with:
# team = input("Enter a team abbreviation (e.g., SAS, BOS, DEN): ").upper()

report_1 = team_scouting_report(team)

## Feature 2: Duo vs Duo

**Purpose:** compare any two duos head-to-head and get a verdict on which one is more efficient and why.

**How it works:**
1. **Input:** two duo names (from the `Lineups` column) or two team abbreviations (uses each team's best duo)
2. **Dataset:** both duos' stats and league ranks under all four models
3. **BALLDONTLIE API:** profiles of all four players (position, height, draft year, country)
4. **OpenAI:** writes the verdict

In [ ]:
def find_duo(name_or_team):
    # A team abbreviation returns that team's best duo; otherwise look up the duo by name
    if name_or_team in list(duos['TEAM']):
        return duos[duos['TEAM'] == name_or_team].sort_values('DES_Balanced', ascending=False).iloc[0]
    return duos[duos['Lineups'] == name_or_team].iloc[0]


def duo_vs_duo(duo_a_input, duo_b_input):
    a = find_duo(duo_a_input)
    b = find_duo(duo_b_input)

    # 1. Dataset: side-by-side stats
    stats = ['MIN', 'OffRtg', 'DefRtg', 'NetRtg', 'TS%', 'AST/TO', 'REB%', 'PIE', 'DES_Balanced', 'DES_Offense', 'DES_Defense', 'DES_Impact']
    compare = pd.DataFrame({a['Lineups'] + ' (' + a['TEAM'] + ')': a[stats], b['Lineups'] + ' (' + b['TEAM'] + ')': b[stats]})
    display(compare)

    # Chart: the four model scores side by side
    scores = compare.loc[['DES_Balanced', 'DES_Offense', 'DES_Defense', 'DES_Impact']].reset_index()
    scores = scores.melt(id_vars=['index'], var_name='Duo', value_name='Score')
    scores['index'] = scores['index'].str.replace('DES_', '')
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=scores, x='index', y='Score', hue='Duo', ax=ax)
    ax.set_xlabel('Model')
    ax.set_ylabel('Duo Efficiency Score (0-100)')
    ax.set_title('Duo vs Duo: score under each model')
    plt.show()

    # 2. API: player profiles
    profiles = []
    for duo in [a, b]:
        for player in [duo['Player_1'], duo['Player_2']]:
            info = get_player(player, duo['TEAM'])
            if info is not None:
                profiles.append(info)
    profiles = pd.DataFrame(profiles)
    display(profiles)

    # 3. OpenAI: verdict
    prompt = f"""
    Compare these two NBA duos from the 2025-26 season. DES = Duo Efficiency Score (0-100, higher is better)
    under four weighting models. Net Rating = points per 100 possessions.

    {compare.to_string()}

    Player profiles:
    {profiles.to_string(index=False)}

    In about 120 words: which duo is more efficient overall, where each one is stronger (offense, defense,
    rebounding, shooting), how the player types explain it, and whether the answer changes by model.
    """
    verdict = get_completion(prompt)
    print(verdict)
    return verdict

In [ ]:
duo_a = 'SAS'
duo_b = 'DEN'
# Or replace these with:
# duo_a = input("First duo (e.g., 'A. Gordon - N. Jokić') or team abbreviation: ")
# duo_b = input("Second duo or team abbreviation: ")

verdict_2 = duo_vs_duo(duo_a, duo_b)

## Feature 3: Build Your Own Model

**Purpose:** the user decides what "efficient" means by choosing the stat weights. The app re-scores every duo, shows how the rankings change compared with our Balanced model, explains the result, and creates an award poster for the winner.

**How it works:**
1. **Input:** a weight for each stat (any positive numbers; they are scaled to add up to 1)
2. **Dataset:** all 2025-26 duos re-scored with `MinMaxScaler` and the user's weights
3. **BALLDONTLIE API:** the winning team's full name, conference and division
4. **OpenAI:** explains the result (text) and generates a "Duo of the Year" poster (image)

In [ ]:
def build_your_own_model(user_weights):
    # 1. Normalize the weights so they add up to 1
    total = sum(user_weights.values())
    weights = {}
    for stat, w in user_weights.items():
        weights[stat] = w / total

    # 2. Dataset: scale the stats to 0-1 and re-score every duo
    score_stats = list(weights.keys())
    scaled = pd.DataFrame(MinMaxScaler().fit_transform(duos[score_stats]), columns=score_stats)
    if 'DefRtg' in score_stats:
        scaled['DefRtg'] = 1 - scaled['DefRtg']     # lower DefRtg is better

    custom = duos.copy()
    custom['DES_Custom'] = 0
    for stat, w in weights.items():
        custom['DES_Custom'] = custom['DES_Custom'] + scaled[stat] * w * 100

    custom['Rank_Custom'] = custom['DES_Custom'].rank(method='min', ascending=False)
    custom['Rank_Balanced'] = custom['DES_Balanced'].rank(method='min', ascending=False)
    custom['Rank_Change'] = custom['Rank_Balanced'] - custom['Rank_Custom']
    top10 = custom.sort_values('DES_Custom', ascending=False).head(10)

    print('Your weights:', {k: round(v, 2) for k, v in weights.items()})
    display(top10[['Lineups', 'TEAM', 'MIN', 'NetRtg', 'DES_Custom', 'Rank_Custom', 'Rank_Balanced', 'Rank_Change']].round(1))

    # Chart: rank under your model vs the Balanced model
    top10_long = top10.melt(id_vars=['Lineups'], value_vars=['Rank_Custom', 'Rank_Balanced'], var_name='Model', value_name='League_Rank')
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(data=top10_long, x='League_Rank', y='Lineups', hue='Model', ax=ax)
    ax.set_xlabel('League rank (1 = best, lower is better)')
    ax.set_ylabel('')
    ax.set_title('Your top 10: rank under your model vs the Balanced model')
    plt.show()

    # 3. API: details on the winning team
    winner = top10.iloc[0]
    team_info = teams[teams['abbreviation'] == winner['TEAM']].iloc[0]

    # 4a. OpenAI: explanation
    prompt = f"""
    A user built their own NBA duo efficiency model with these stat weights (they add up to 1):
    {weights}
    Stats: NetRtg = net rating, OffRtg = offensive rating, DefRtg = defensive rating (lower is better, so it is flipped),
    TS% = true shooting, AST/TO = assist-to-turnover ratio, REB% = rebound rate, PIE = player impact estimate.

    The new top 10 (Rank_Change = places gained compared with our Balanced model):
    {top10[['Lineups', 'TEAM', 'NetRtg', 'DES_Custom', 'Rank_Balanced', 'Rank_Change']].round(1).to_string(index=False)}

    In about 120 words: what kind of duo this model rewards, which duos rose or fell the most and why,
    and why {winner['Lineups']} of the {team_info['full_name']} comes out on top.
    """
    explanation = get_completion(prompt)
    print(explanation)

    # 4b. OpenAI: award poster (no real people, just the award design)
    image_prompt = (f"A bold sports award poster titled 'DUO OF THE YEAR', with the subtitle '{team_info['full_name']}'. "
                    f"Two basketballs side by side on a glowing hardwood court under arena lights, a gold trophy between them, "
                    f"in the {team_info['full_name']} team colors. No people. Clean modern graphic design.")
    display(Image(data=generate_image(image_prompt)))
    return explanation

In [ ]:
# Example: a user who values defense and rebounding most
my_weights = {'DefRtg': 5, 'REB%': 3, 'NetRtg': 2, 'TS%': 1}
# Or replace this with input, e.g.:
# my_weights = {'DefRtg': float(input("Weight for DefRtg: ")), 'REB%': float(input("Weight for REB%: ")),
#               'NetRtg': float(input("Weight for NetRtg: ")), 'TS%': float(input("Weight for TS%: "))}

explanation_3 = build_your_own_model(my_weights)

## Summary
- **Feature 1** combines our scores, live team results, and OpenAI text + audio into a scouting report for any team.
- **Feature 2** compares any two duos across all four models, adds live player profiles, and has OpenAI explain the verdict.
- **Feature 3** lets the user define efficiency, re-ranks every duo, and uses OpenAI to explain the result and design an award poster.
